In [1]:
import glob
import subprocess as sb
import os
import ntpath
import re
import shutil

In [3]:

# Rscript STOmics_seurat.only_gem2rds.R -i <input.gem> -b <binsize> -s <sample name> -o <outdir>

# Rscript STOmics_seurat.only_gem2rds.R -i /cluster/huanglab/dhong/project/MouseST/Data/Stereo/GEM/CD1-e12-5-13-5.tissue.gem.gz -b <binsize> -s <sample name> -o <outdir>

# Gets all matching file 
def get_files(paths):
    file_ins = []
    for path in paths:
        file_ins.extend(glob.glob(path + '*tissue.gem.gz'))
    file_ins = sorted(file_ins)
    return file_ins

# input file path
paths = [
    '/cluster/huanglab/lab/LabDataset/jiamao/HumanSpinalCord/RawData/STOmics/GW7_98729/GW7/', # B03210A111.tissue.gem.gz
    '/cluster/huanglab/lab/LabDataset/jiamao/HumanSpinalCord/RawData/STOmics/GW12_98802/GW12/', # B03210E111.tissue.gem.gz
    '/cluster/huanglab/lab/LabDataset/dhong/HumanSpinalCord/RawData/STOmics/GW9_ST/01.StandardWorkflow_Result/GeneExpMatrix/', # B01318A2.tissue.gem.gz
    '/cluster/huanglab/lab/LabDataset/dhong/HumanSpinalCord/RawData/STOmics/GW10_ST/01.StandardWorkflow_Result/GeneExpMatrix/', # D01869D5.tissue.gem.gz
    '/cluster/huanglab/lab/LabDataset/dhong/HumanSpinalCord/RawData/STOmics/GW16_ST/01.StandardWorkflow_Result/GeneExpMatrix/', # D01869D2.tissue.gem.gz
    '/cluster/huanglab/lab/LabDataset/dhong/HumanSpinalCord/RawData/STOmics/GW8_GW17_ST_report/', # GW8.tissue.gem.gz; GW17.tissue.gem.gz
    '/cluster/huanglab/lab/LabDataset/dhong/HumanSpinalCord/RawData/STOmics/Y3-1_ST/01.StandardWorkflow_Result/GeneExpMatrix/', # D01567D5.tissue.gef
    '/cluster/huanglab/lab/LabDataset/dhong/HumanSpinalCord/RawData/STOmics/Y3-2_ST/01.StandardWorkflow_Result/GeneExpMatrix/' # D01567F2.tissue.gef
]

# get flie path
file_ins = get_files(paths)

# output file path
print('\n'.join(file_ins))

/cluster/huanglab/lab/LabDataset/dhong/HumanSpinalCord/RawData/STOmics/GW10_ST/01.StandardWorkflow_Result/GeneExpMatrix/D01869D5.tissue.gem.gz
/cluster/huanglab/lab/LabDataset/dhong/HumanSpinalCord/RawData/STOmics/GW16_ST/01.StandardWorkflow_Result/GeneExpMatrix/D01869D2.tissue.gem.gz
/cluster/huanglab/lab/LabDataset/dhong/HumanSpinalCord/RawData/STOmics/GW8_GW17_ST_report/GW17.tissue.gem.gz
/cluster/huanglab/lab/LabDataset/dhong/HumanSpinalCord/RawData/STOmics/GW8_GW17_ST_report/GW8.tissue.gem.gz
/cluster/huanglab/lab/LabDataset/dhong/HumanSpinalCord/RawData/STOmics/GW9_ST/01.StandardWorkflow_Result/GeneExpMatrix/B01318A2.tissue.gem.gz
/cluster/huanglab/lab/LabDataset/dhong/HumanSpinalCord/RawData/STOmics/Y3-1_ST/01.StandardWorkflow_Result/GeneExpMatrix/D01567D5.tissue.gem.gz
/cluster/huanglab/lab/LabDataset/dhong/HumanSpinalCord/RawData/STOmics/Y3-2_ST/01.StandardWorkflow_Result/GeneExpMatrix/D01567F2.tissue.gem.gz
/cluster/huanglab/lab/LabDataset/jiamao/HumanSpinalCord/RawData/STOmi

In [ ]:
# copy file to 
copy_dir = '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/tissue_gem/'

if not os.path.exists(copy_dir):
    os.makedirs(copy_dir)

# rename & correct
rename_map = {
    'B03210A111': 'GW6', # Second batch
    'B03210E111': 'GW12', # Second batch
    'B01318A2': 'GW9',
    'D01869D5': 'GW16', # Note that this is GW16, but it is under the folder of GW10 because the company’s sequencing name is reversed.
    'D01869D2': 'GW10', # Note that this is GW10, but it is under the folder of GW16 because the company’s sequencing name is reversed.
    'GW8': 'GW7', # Note that this is GW7, but it was mistakenly thought to be GW8 when the sample was sent, and it was initially considered GW6 in the first round of analysis because it was the earliest sample, but after the second round of analysis included the GW7 sample, it was found that this sample be GW7.
    'GW17': 'GW17',
    'D01567D5': 'Y31',
    'D01567F2': 'Y32'
}

# copy and rename
for file_path in file_ins:
    # sample ID
    file_name = os.path.basename(file_path)
    sample = file_name.split('.')
    
    # remap
    copy_name = rename_map[sample[0]] + '.tissue.gem.gz'
    
    # output
    copy_file_path = os.path.join(copy_dir, copy_name)
    shutil.copy(file_path, copy_file_path)

In [ ]:
# prepare
gem2rds='/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/code/STOmics_seurat.only_gem2rds.R'

sample_in = '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/tissue_gem/'
sample_ins = sorted(glob.glob(sample_in+'*tissue.gem.gz'))

In [8]:
# bin50
sample_out = '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin50Data/'
if not os.path.exists(sample_out):
    os.makedirs(sample_out)

for sample_in in sample_ins:
    basename = ntpath.basename(sample_in)
    file_out = basename.replace('.tissue.gem.gz','')
    
    cmd = "Rscript " + gem2rds + ' -i ' + sample_in + ' -b 50' +  ' -s '+ file_out +' -o ' + sample_out  \
   
    print(cmd)
    sb.call(cmd,shell=True)

print('Yeh, all done!!!')

Rscript /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/code/STOmics_seurat.only_gem2rds.R -i /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/tissue_gem/GW10.tissue.gem.gz -b 50 -s GW10 -o /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin50Data/
[1] "/cluster2/huanglab/jiamao/conda/envs/R-4.3.0/lib/R/library"


Attaching SeuratObject
‘SeuratObject’ was built under R 4.3.1 but the current version is
4.3.2; it is recomended that you reinstall ‘SeuratObject’ as the ABI
for R may have changed
Seurat v4 was just loaded with SeuratObject v5; disabling v5 assays and
validation routines, and ensuring assays work in strict v3/v4
compatibility mode
Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat
Warning message:
In dir.create(opts$outdir, recursive = TRUE) :
  '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin50Data' already exists


Rscript /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/code/STOmics_seurat.only_gem2rds.R -i /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/tissue_gem/GW12.tissue.gem.gz -b 50 -s GW12 -o /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin50Data/
[1] "/cluster2/huanglab/jiamao/conda/envs/R-4.3.0/lib/R/library"


Attaching SeuratObject
‘SeuratObject’ was built under R 4.3.1 but the current version is
4.3.2; it is recomended that you reinstall ‘SeuratObject’ as the ABI
for R may have changed
Seurat v4 was just loaded with SeuratObject v5; disabling v5 assays and
validation routines, and ensuring assays work in strict v3/v4
compatibility mode
Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat
Warning message:
In dir.create(opts$outdir, recursive = TRUE) :
  '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin50Data' already exists


Rscript /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/code/STOmics_seurat.only_gem2rds.R -i /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/tissue_gem/GW16.tissue.gem.gz -b 50 -s GW16 -o /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin50Data/
[1] "/cluster2/huanglab/jiamao/conda/envs/R-4.3.0/lib/R/library"


...

Rscript /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/code/STOmics_seurat.only_gem2rds.R -i /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/tissue_gem/Y31.tissue.gem.gz -b 50 -s Y31 -o /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin50Data/
[1] "/cluster2/huanglab/jiamao/conda/envs/R-4.3.0/lib/R/library"


Attaching SeuratObject
‘SeuratObject’ was built under R 4.3.1 but the current version is
4.3.2; it is recomended that you reinstall ‘SeuratObject’ as the ABI
for R may have changed
Seurat v4 was just loaded with SeuratObject v5; disabling v5 assays and
validation routines, and ensuring assays work in strict v3/v4
compatibility mode
Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat
Warning message:
In dir.create(opts$outdir, recursive = TRUE) :
  '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin50Data' already exists


Rscript /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/code/STOmics_seurat.only_gem2rds.R -i /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/tissue_gem/Y32.tissue.gem.gz -b 50 -s Y32 -o /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin50Data/
[1] "/cluster2/huanglab/jiamao/conda/envs/R-4.3.0/lib/R/library"


Attaching SeuratObject
‘SeuratObject’ was built under R 4.3.1 but the current version is
4.3.2; it is recomended that you reinstall ‘SeuratObject’ as the ABI
for R may have changed
Seurat v4 was just loaded with SeuratObject v5; disabling v5 assays and
validation routines, and ensuring assays work in strict v3/v4
compatibility mode
Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat
Warning message:
In dir.create(opts$outdir, recursive = TRUE) :
  '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin50Data' already exists


Yeh, all done!!!


In [9]:
# bin20
sample_out = '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin20Data/'
if not os.path.exists(sample_out):
    os.makedirs(sample_out)

for sample_in in sample_ins:
    basename = ntpath.basename(sample_in)
    file_out = basename.replace('.tissue.gem.gz','')
    
    cmd = "Rscript " + gem2rds + ' -i ' + sample_in + ' -b 20' +  ' -s '+ file_out +' -o ' + sample_out  \
   
    print(cmd)
    sb.call(cmd,shell=True)

print('Yeh, all done!!!')

Rscript /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/code/STOmics_seurat.only_gem2rds.R -i /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/tissue_gem/GW10.tissue.gem.gz -b 20 -s GW10 -o /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin20Data/
[1] "/cluster2/huanglab/jiamao/conda/envs/R-4.3.0/lib/R/library"


Attaching SeuratObject
‘SeuratObject’ was built under R 4.3.1 but the current version is
4.3.2; it is recomended that you reinstall ‘SeuratObject’ as the ABI
for R may have changed
Seurat v4 was just loaded with SeuratObject v5; disabling v5 assays and
validation routines, and ensuring assays work in strict v3/v4
compatibility mode
Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat
Warning message:
In dir.create(opts$outdir, recursive = TRUE) :
  '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin20Data' already exists


Rscript /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/code/STOmics_seurat.only_gem2rds.R -i /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/tissue_gem/GW12.tissue.gem.gz -b 20 -s GW12 -o /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin20Data/
[1] "/cluster2/huanglab/jiamao/conda/envs/R-4.3.0/lib/R/library"


Attaching SeuratObject
‘SeuratObject’ was built under R 4.3.1 but the current version is
4.3.2; it is recomended that you reinstall ‘SeuratObject’ as the ABI
for R may have changed
Seurat v4 was just loaded with SeuratObject v5; disabling v5 assays and
validation routines, and ensuring assays work in strict v3/v4
compatibility mode
Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat
Warning message:
In dir.create(opts$outdir, recursive = TRUE) :
  '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin20Data' already exists


Rscript /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/code/STOmics_seurat.only_gem2rds.R -i /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/tissue_gem/GW16.tissue.gem.gz -b 20 -s GW16 -o /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin20Data/
[1] "/cluster2/huanglab/jiamao/conda/envs/R-4.3.0/lib/R/library"


...

Rscript /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/code/STOmics_seurat.only_gem2rds.R -i /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/tissue_gem/Y31.tissue.gem.gz -b 20 -s Y31 -o /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin20Data/
[1] "/cluster2/huanglab/jiamao/conda/envs/R-4.3.0/lib/R/library"


Attaching SeuratObject
‘SeuratObject’ was built under R 4.3.1 but the current version is
4.3.2; it is recomended that you reinstall ‘SeuratObject’ as the ABI
for R may have changed
Seurat v4 was just loaded with SeuratObject v5; disabling v5 assays and
validation routines, and ensuring assays work in strict v3/v4
compatibility mode
Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat
Warning message:
In dir.create(opts$outdir, recursive = TRUE) :
  '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin20Data' already exists


Rscript /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/code/STOmics_seurat.only_gem2rds.R -i /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/tissue_gem/Y32.tissue.gem.gz -b 20 -s Y32 -o /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin20Data/
[1] "/cluster2/huanglab/jiamao/conda/envs/R-4.3.0/lib/R/library"


Attaching SeuratObject
‘SeuratObject’ was built under R 4.3.1 but the current version is
4.3.2; it is recomended that you reinstall ‘SeuratObject’ as the ABI
for R may have changed
Seurat v4 was just loaded with SeuratObject v5; disabling v5 assays and
validation routines, and ensuring assays work in strict v3/v4
compatibility mode
Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat
Warning message:
In dir.create(opts$outdir, recursive = TRUE) :
  '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin20Data' already exists


Yeh, all done!!!


In [10]:
# bin30
sample_out = '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin30Data/'
if not os.path.exists(sample_out):
    os.makedirs(sample_out)

for sample_in in sample_ins:
    basename = ntpath.basename(sample_in)
    file_out = basename.replace('.tissue.gem.gz','')
    
    cmd = "Rscript " + gem2rds + ' -i ' + sample_in + ' -b 30' +  ' -s '+ file_out +' -o ' + sample_out  \
   
    print(cmd)
    sb.call(cmd,shell=True)

print('Yeh, all done!!!')

Rscript /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/code/STOmics_seurat.only_gem2rds.R -i /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/tissue_gem/GW10.tissue.gem.gz -b 30 -s GW10 -o /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin30Data/
[1] "/cluster2/huanglab/jiamao/conda/envs/R-4.3.0/lib/R/library"


Attaching SeuratObject
‘SeuratObject’ was built under R 4.3.1 but the current version is
4.3.2; it is recomended that you reinstall ‘SeuratObject’ as the ABI
for R may have changed
Seurat v4 was just loaded with SeuratObject v5; disabling v5 assays and
validation routines, and ensuring assays work in strict v3/v4
compatibility mode
Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat
Warning message:
In dir.create(opts$outdir, recursive = TRUE) :
  '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin30Data' already exists


Rscript /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/code/STOmics_seurat.only_gem2rds.R -i /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/tissue_gem/GW12.tissue.gem.gz -b 30 -s GW12 -o /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin30Data/
[1] "/cluster2/huanglab/jiamao/conda/envs/R-4.3.0/lib/R/library"


Attaching SeuratObject
‘SeuratObject’ was built under R 4.3.1 but the current version is
4.3.2; it is recomended that you reinstall ‘SeuratObject’ as the ABI
for R may have changed
Seurat v4 was just loaded with SeuratObject v5; disabling v5 assays and
validation routines, and ensuring assays work in strict v3/v4
compatibility mode
Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat
Warning message:
In dir.create(opts$outdir, recursive = TRUE) :
  '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin30Data' already exists


Rscript /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/code/STOmics_seurat.only_gem2rds.R -i /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/tissue_gem/GW16.tissue.gem.gz -b 30 -s GW16 -o /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin30Data/
[1] "/cluster2/huanglab/jiamao/conda/envs/R-4.3.0/lib/R/library"


...

Rscript /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/code/STOmics_seurat.only_gem2rds.R -i /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/tissue_gem/Y31.tissue.gem.gz -b 30 -s Y31 -o /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin30Data/
[1] "/cluster2/huanglab/jiamao/conda/envs/R-4.3.0/lib/R/library"


Attaching SeuratObject
‘SeuratObject’ was built under R 4.3.1 but the current version is
4.3.2; it is recomended that you reinstall ‘SeuratObject’ as the ABI
for R may have changed
Seurat v4 was just loaded with SeuratObject v5; disabling v5 assays and
validation routines, and ensuring assays work in strict v3/v4
compatibility mode
Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat
Warning message:
In dir.create(opts$outdir, recursive = TRUE) :
  '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin30Data' already exists


Rscript /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/code/STOmics_seurat.only_gem2rds.R -i /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/tissue_gem/Y32.tissue.gem.gz -b 30 -s Y32 -o /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin30Data/
[1] "/cluster2/huanglab/jiamao/conda/envs/R-4.3.0/lib/R/library"


Attaching SeuratObject
‘SeuratObject’ was built under R 4.3.1 but the current version is
4.3.2; it is recomended that you reinstall ‘SeuratObject’ as the ABI
for R may have changed
Seurat v4 was just loaded with SeuratObject v5; disabling v5 assays and
validation routines, and ensuring assays work in strict v3/v4
compatibility mode
Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat
Warning message:
In dir.create(opts$outdir, recursive = TRUE) :
  '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/data/bin30Data' already exists


Yeh, all done!!!
